##### **Setup**

In [173]:
import pandas as pd
import numpy as np

df = pd.read_stata('data.dta', convert_categoricals=False)
print("Shape:", df.shape)
print("Unique participants:", df['participantid'].nunique())
print("Forward-filled rows total:", len(df))

Shape: (956, 1178)
Unique participants: 55
Forward-filled rows total: 956


##### **Cronbach's alpha helper**

In [174]:
def cronbach_alpha(d):
    """Cronbach's alpha for a DataFrame where columns are items and rows are cases."""
    d = d.dropna()
    k = d.shape[1]
    item_var = d.var(axis=0, ddof=1).sum()
    total_var = d.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_var / total_var)

#### **Score UCLA (trait loneliness)**

In [175]:
UCLA_MAP = {
    'Stimmt gar nicht':   1,
    'Stimmt wenig':       2,
    'Stimmt teils-teils': 3,
    'Stimmt ziemlich':    4,
    'Stimmt völlig':      5,
}

# UCLA is asked once at baseline; in the forward-filled file every row of a
# participant carries the same answer, so take one row per person.
person = df.drop_duplicates('participantid').copy()

# Map German strings -> numbers
for i in range(1, 21):
    person[f'einsam_{i}'] = person[f'einsam_{i}'].map(UCLA_MAP)

# Data-driven reversal: split the 20 items into two polarity groups via the
# first principal component, then reverse the group with HIGHER mean
# (in a student sample, positively-worded items have higher means).
items = person[[f'einsam_{i}' for i in range(1, 21)]]
eigvals, eigvecs = np.linalg.eigh(items.corr().values)
pc1 = eigvecs[:, -1]

group_A = [i for i, l in enumerate(pc1, start=1) if l > 0]
group_B = [i for i, l in enumerate(pc1, start=1) if l < 0]
mean_A = items[[f'einsam_{i}' for i in group_A]].mean().mean()
mean_B = items[[f'einsam_{i}' for i in group_B]].mean().mean()
ucla_reverse = group_A if mean_A > mean_B else group_B

print(f"UCLA items reversed: {sorted(ucla_reverse)} ({len(ucla_reverse)} items)")

# Apply reversal
for i in ucla_reverse:
    person[f'einsam_{i}'] = 6 - person[f'einsam_{i}']

# Sum to get UCLA total
einsam_cols = [f'einsam_{i}' for i in range(1, 21)]
person['UCLA_total'] = person[einsam_cols].sum(axis=1)

alpha = cronbach_alpha(person[einsam_cols])
print(f"\nUCLA total: N={len(person)}, "
      f"M={person['UCLA_total'].mean():.2f}, "
      f"SD={person['UCLA_total'].std():.2f}, "
      f"range=[{person['UCLA_total'].min():.0f}, {person['UCLA_total'].max():.0f}], "
      f"alpha={alpha:.3f}")

# Merge UCLA_total back into the main (forward-filled) df
df = df.merge(person[['participantid', 'UCLA_total']],
              on='participantid', how='left')
print(f"\ndf shape after merge: {df.shape}")
print(f"UCLA_total non-null in df: {df['UCLA_total'].notna().sum()} of {len(df)}")

UCLA items reversed: [1, 2, 4, 5, 8, 9, 14, 15, 18, 19] (10 items)

UCLA total: N=55, M=39.98, SD=11.55, range=[22, 71], alpha=0.912

df shape after merge: (956, 1179)
UCLA_total non-null in df: 956 of 956


##### **Score BSSS (perceived social support)**
BSSS is also a baseline measure (asked once, forward-filled). All 8 items are positively worded ("There are people who really like me," "I have people I can always rely on," etc.), so no reverse-coding is needed — high score = more perceived support.

In [176]:
BSSS_MAP = {
    'Stimme überhaupt nicht zu': 1,
    'Stimme eher nicht zu':      2,
    'Stimme etwas zu':           3,
    'Stimme voll und ganz zu':   4,
}

# BSSS is a baseline measure — one row per participant
person = df.drop_duplicates('participantid').copy()
for i in range(1, 9):
    person[f'social_support_{i}'] = person[f'social_support_{i}'].map(BSSS_MAP)

bsss_cols = [f'social_support_{i}' for i in range(1, 9)]

# All 8 BSSS items are positively worded ("there are people who like me",
# "I have people I can rely on", ...). High = more support. No reversal needed.
# Sanity check: every item should have high mean and positive correlations.
print("Item means (1-4 scale):")
print(person[bsss_cols].mean().round(2).to_string())
corr_upper = person[bsss_cols].corr().values[np.triu_indices(8, k=1)]
print(f"\nInter-item correlations: min={corr_upper.min():.2f}, max={corr_upper.max():.2f}")

# Sum and mean (both useful — sum for raw score, mean for 1-4 interpretable scale)
person['BSSS_total'] = person[bsss_cols].sum(axis=1)
person['BSSS_mean']  = person[bsss_cols].mean(axis=1)

alpha = cronbach_alpha(person[bsss_cols])
print(f"\nBSSS:  N={len(person)}, "
      f"sum M={person['BSSS_total'].mean():.2f}, SD={person['BSSS_total'].std():.2f}, "
      f"mean M={person['BSSS_mean'].mean():.2f}, alpha={alpha:.3f}")

# Merge back into main df
df = df.merge(person[['participantid', 'BSSS_total', 'BSSS_mean']],
              on='participantid', how='left')
print(f"\ndf shape after merge: {df.shape}")

Item means (1-4 scale):
social_support_1    3.76
social_support_2    3.49
social_support_3    3.65
social_support_4    3.56
social_support_5    3.69
social_support_6    3.60
social_support_7    3.65
social_support_8    3.25

Inter-item correlations: min=0.28, max=0.69

BSSS:  N=55, sum M=28.67, SD=3.85, mean M=3.58, alpha=0.872

df shape after merge: (956, 1181)


#### **Score PHQ-9 (depression)**
PHQ-9 is a baseline measure, 9 items, all measuring depressive symptoms (so all negatively worded — high score = more depressed). No reverse-coding. Per Supervisor's guidance: use SUM scoring. One catch — the data has empty strings '' for missing values that need to be treated as NaN.

Just one participant (110982) missing exactly one item: depression_9 (the suicide/self-harm question — which is the most commonly skipped PHQ item in any sample because participants often leave it blank).

Standard PHQ-9 scoring practice (Kroenke & Spitzer 2001): if ≤ 2 items are missing, prorate the score using the mean of completed items times 9. This keeps the participant in. 

In [177]:
PHQ9_MAP = {
    'Überhaupt nicht':                0,
    'An einzelnen Tagen':             1,
    'An mehr als der Hälfte der Tage': 2,
    'Beinahe jeden Tag':              3,
    '':                                np.nan,
}

# Baseline measure — one row per participant
person = df.drop_duplicates('participantid').copy()
for i in range(1, 10):
    person[f'depression_{i}'] = person[f'depression_{i}'].map(PHQ9_MAP)

phq_cols = [f'depression_{i}' for i in range(1, 10)]

# All 9 items are negatively worded (depressive symptoms). High = more depressed.
# No reverse-coding. SUM scoring per official PHQ-9 instructions (Heike confirmed).
print("Item means (0-3 scale):")
print(person[phq_cols].mean().round(2).to_string())

# Score with proration: if a participant has at most 2 missing items, impute by
# (mean of completed) * 9. If 3+ missing, score is NaN.
n_complete = person[phq_cols].notna().sum(axis=1)
mean_completed = person[phq_cols].mean(axis=1)
person['PHQ9_total'] = np.where(n_complete >= 7,
                                 mean_completed * 9,
                                 np.nan)

n_valid = person['PHQ9_total'].notna().sum()
alpha = cronbach_alpha(person[phq_cols])
print(f"\nPHQ-9: N={n_valid} (of {len(person)}), "
      f"M={person['PHQ9_total'].mean():.2f}, "
      f"SD={person['PHQ9_total'].std():.2f}, "
      f"range=[{person['PHQ9_total'].min():.1f}, {person['PHQ9_total'].max():.1f}], "
      f"alpha={alpha:.3f}")
print(f"Note: 1 participant (110982) had depression_9 missing — prorated.")

# Merge into main df
df = df.merge(person[['participantid', 'PHQ9_total']], on='participantid', how='left')
print(f"\ndf shape after merge: {df.shape}")

Item means (0-3 scale):
depression_1    1.16
depression_2    1.09
depression_3    1.36
depression_4    1.53
depression_5    0.85
depression_6    1.11
depression_7    1.47
depression_8    0.60
depression_9    0.24

PHQ-9: N=55 (of 55), M=9.43, SD=4.52, range=[1.0, 20.0], alpha=0.756
Note: 1 participant (110982) had depression_9 missing — prorated.

df shape after merge: (956, 1182)


##### **Score GAD-7 (anxiety)**

In [178]:
GAD7_MAP = PHQ9_MAP   # exact same German strings and 0–3 numeric scale

# Baseline measure — one row per participant
person = df.drop_duplicates('participantid').copy()
for i in range(1, 8):
    person[f'anxiety_{i}'] = person[f'anxiety_{i}'].map(GAD7_MAP)

gad_cols = [f'anxiety_{i}' for i in range(1, 8)]

# All 7 items are negatively worded (anxiety symptoms). High = more anxious.
# Sum scoring. Prorate if up to 2 items missing (≥ 5 complete needed).
print("Item means (0-3 scale):")
print(person[gad_cols].mean().round(2).to_string())

n_complete = person[gad_cols].notna().sum(axis=1)
mean_completed = person[gad_cols].mean(axis=1)
person['GAD7_total'] = np.where(n_complete >= 5,
                                 mean_completed * 7,
                                 np.nan)

n_valid = person['GAD7_total'].notna().sum()
alpha = cronbach_alpha(person[gad_cols])
print(f"\nGAD-7: N={n_valid} (of {len(person)}), "
      f"M={person['GAD7_total'].mean():.2f}, "
      f"SD={person['GAD7_total'].std():.2f}, "
      f"range=[{person['GAD7_total'].min():.1f}, {person['GAD7_total'].max():.1f}], "
      f"alpha={alpha:.3f}")

# Merge
df = df.merge(person[['participantid', 'GAD7_total']], on='participantid', how='left')
print(f"\ndf shape after merge: {df.shape}")

Item means (0-3 scale):
anxiety_1    1.36
anxiety_2    1.16
anxiety_3    1.44
anxiety_4    1.27
anxiety_5    0.87
anxiety_6    0.82
anxiety_7    0.84

GAD-7: N=55 (of 55), M=7.76, SD=4.29, range=[1.0, 18.0], alpha=0.787

df shape after merge: (956, 1183)


#### **Score Mood (PA, NA, somatic)**
Mood is the daily-diary measure (956 rows, no deduplication). Each mood_N is a 0–100 slider value, already numeric. Per the Fragebogen:

mood_1 stressed, mood_3 nervous, mood_7 sad, mood_8 dejected, mood_9 concerned → Negative Affect (NA)

mood_2 balanced, mood_4 happy, mood_5 enthusiastic, mood_6 confident → Positive Affect (PA)

mood_10 tired, mood_11 sick → somatic (separate from affect )

In [179]:
# Mood is a daily-diary (within-person) measure. Already numeric 0–100 slider,
# so no string-to-number mapping needed. Work on all 956 rows directly.

PA_items = [2, 4, 5, 6]      # balanced, happy, enthusiastic, confident
NA_items = [1, 3, 7, 8, 9]   # stressed, nervous, sad, dejected, concerned
# mood_10 (tired) and mood_11 (sick) are somatic items, NOT affect.
# Kept as standalone variables, not combined into a scale.

print("Item means (0-100):")
print(df[[f'mood_{i}' for i in range(1, 12)]].mean().round(2).to_string())

df['mood_PA'] = df[[f'mood_{i}' for i in PA_items]].mean(axis=1)
df['mood_NA'] = df[[f'mood_{i}' for i in NA_items]].mean(axis=1)

n_obs = df['mood_PA'].notna().sum()
alpha_PA = cronbach_alpha(df[[f'mood_{i}' for i in PA_items]])
alpha_NA = cronbach_alpha(df[[f'mood_{i}' for i in NA_items]])

print(f"\nValid mood observations: {n_obs} of {len(df)}")
print(f"PA (positive affect): M={df['mood_PA'].mean():.2f}, SD={df['mood_PA'].std():.2f}, alpha={alpha_PA:.3f}")
print(f"NA (negative affect): M={df['mood_NA'].mean():.2f}, SD={df['mood_NA'].std():.2f}, alpha={alpha_NA:.3f}")

# ICC(1): share of variance that is between-person (trait) vs within-person (state).
# Tells us whether PA/NA is mostly a stable trait or fluctuates day-to-day.
def icc1(d, var, id_col='participantid'):
    person_means = d.groupby(id_col)[var].mean()
    var_between = person_means.var(ddof=1)
    var_within = d.groupby(id_col)[var].var(ddof=1).mean()
    return var_between / (var_between + var_within)

print(f"\nICC(1) between-person share of variance:")
print(f"  PA: ICC={icc1(df, 'mood_PA'):.2f}")
print(f"  NA: ICC={icc1(df, 'mood_NA'):.2f}")
print(f"\ndf shape: {df.shape}")

Item means (0-100):
mood_1     26.66
mood_2     48.46
mood_3     18.70
mood_4     54.47
mood_5     31.40
mood_6     49.84
mood_7     14.79
mood_8     20.09
mood_9     27.30
mood_10    39.64
mood_11    14.22

Valid mood observations: 956 of 956
PA (positive affect): M=46.04, SD=20.32, alpha=0.826
NA (negative affect): M=21.51, SD=18.65, alpha=0.841

ICC(1) between-person share of variance:
  PA: ICC=0.53
  NA: ICC=0.51

df shape: (956, 1185)


#### **State (momentary) loneliness**
This is the diary-level loneliness measure: 8 items, 0–4 scale. Per the Fragebogen, items 1–4 are negative ("feel alone", etc.) and items 5–8 are positive ("understand me", "have enough company") — so {5, 6, 7, 8} would need reversal.

In [180]:
LON_MAP = {'0 gar nicht': 0, '1': 1, '2': 2, '3': 3, '4 sehr': 4, '': np.nan}

# Build a temporary, polarity-aligned numeric frame. Original df columns left untouched.
items = pd.DataFrame({
    f'loneliness_{i}': df[f'loneliness_{i}'].map(LON_MAP)
    for i in range(1, 9)
})

print("Item means (0-4, before reversal):")
print(items.mean().round(2).to_string())

# Items 1-4 negative ("I feel alone", "I feel excluded", etc.) → keep
# Items 5-8 positive ("people understand me", "have enough company") → reverse
for i in [5, 6, 7, 8]:
    items[f'loneliness_{i}'] = 4 - items[f'loneliness_{i}']

# Mean across the 8 items → state loneliness score, range 0-4, high = more lonely
df['loneliness_state'] = items.mean(axis=1)

n_obs = df['loneliness_state'].notna().sum()
alpha_lon = cronbach_alpha(items)
icc_lon = icc1(df, 'loneliness_state')

print(f"\nState loneliness:")
print(f"  N obs = {n_obs} of {len(df)}")
print(f"  M     = {df['loneliness_state'].mean():.2f}")
print(f"  SD    = {df['loneliness_state'].std():.2f}")
print(f"  alpha = {alpha_lon:.3f}")
print(f"  ICC   = {icc_lon:.2f}  (between-person share of variance)")
print(f"\ndf shape: {df.shape}")

Item means (0-4, before reversal):
loneliness_1    0.71
loneliness_2    0.66
loneliness_3    0.40
loneliness_4    0.53
loneliness_5    3.10
loneliness_6    3.47
loneliness_7    3.40
loneliness_8    3.14

State loneliness:
  N obs = 954 of 956
  M     = 0.65
  SD    = 0.67
  alpha = 0.893
  ICC   = 0.75  (between-person share of variance)

df shape: (956, 1186)


##### **Study stress (last scale)** 
Only 2 items, 0–4 scale. Per Fragebogen:

stress_uni_1: "I am currently achieving the performance required in courses without great effort." → This is a coping item (high = handling things well = NOT stressed) → reverse.

stress_uni_2: "I'm finding it difficult to motivate myself to study at the moment." → This is the stress item (high = more stress) → keep.

For 2-item scales, Cronbach's α isn't really informative — we report the inter-item correlation and Spearman-Brown reliability instead.

In [181]:
STRESS_MAP = {'0 gar nicht': 0, '1': 1, '2': 2, '3': 3, '4 sehr': 4, '': np.nan}

items = pd.DataFrame({
    'stress_uni_1': df['stress_uni_1'].map(STRESS_MAP),  # coping (reverse)
    'stress_uni_2': df['stress_uni_2'].map(STRESS_MAP),  # stress  (keep)
})

print("Item means (0-4, before reversal):")
print(items.mean().round(2).to_string())

# Reverse the coping item so high = more stressed throughout
items['stress_uni_1'] = 4 - items['stress_uni_1']

# Inter-item correlation and Spearman-Brown 2-item reliability
r = items.corr().iloc[0, 1]
sb = 2 * r / (1 + r)

df['stress_uni'] = items.mean(axis=1)
n_obs = df['stress_uni'].notna().sum()
icc = icc1(df, 'stress_uni')

print(f"\nStudy stress:")
print(f"  N obs = {n_obs} of {len(df)}")
print(f"  M     = {df['stress_uni'].mean():.2f}")
print(f"  SD    = {df['stress_uni'].std():.2f}")
print(f"  Inter-item r  = {r:.2f}")
print(f"  Spearman-Brown reliability = {sb:.3f}")
print(f"  ICC = {icc:.2f}")
print(f"\ndf shape: {df.shape}")

Item means (0-4, before reversal):
stress_uni_1    1.91
stress_uni_2    2.39

Study stress:
  N obs = 949 of 956
  M     = 2.24
  SD    = 0.98
  Inter-item r  = 0.44
  Spearman-Brown reliability = 0.616
  ICC = 0.81

df shape: (956, 1187)


#### **Sanity check: validity correlations**

In [182]:
# Take one row per participant (trait-level variables are forward-filled, so any row works)
person = df.drop_duplicates('participantid')[
    ['UCLA_total', 'BSSS_mean', 'PHQ9_total', 'GAD7_total']
].copy()

print("Between-person validity correlations (N=55):")
print(person.corr().round(2).to_string())

Between-person validity correlations (N=55):
            UCLA_total  BSSS_mean  PHQ9_total  GAD7_total
UCLA_total        1.00      -0.67        0.49        0.43
BSSS_mean        -0.67       1.00       -0.42       -0.31
PHQ9_total        0.49      -0.42        1.00        0.76
GAD7_total        0.43      -0.31        0.76        1.00


##### **What we have**
● 459 transcripts from 41 unique participants (not 55 — 14 baseline participants never recorded any audio)

● 3 measurement waves (surveys): 23726, 23727, 23728 — roughly 150 transcripts each wave

● Up to 20 transcripts per participant (max), 4 participants with only 1

Filename structure (two layouts, both valid)
    Format	Count	Example
    Old	    292	    Text <ts>-<32-char UPPERCASE hash>-<pid>-43-<survey>.docx
    New	    167	    Text <ts>-<16-char lowercase hash>-<pid>-<survey>-43.docx


The "43" is the question ID (always 43 — that's the audio-diary question), and the position of survey vs 43 flips between old/new layouts. The hash is the file's unique key — that's what we'll match to the audio URLs in relationship_satisfaction.

Transcript content sample:
File: Text 20250127121404998-098625F7BDC046EE9A4B44101A7AEF00-110918-43-23726.docx

    "Person 1: #00:00:02-7# Ich bin sehr zufrieden von den sozialen Interaktionen, weil ich war beim Fußball und ansonsten danach bin ich krank geworden. Deswegen kann ich da jetzt nicht so viel zu sagen." 
    

Each transcript is one paragraph with:

        ● Speaker tag (Person 1:)

        ● Timestamp markers (#00:00:02-7#)

        ● German speech text

##### **Verify the linkage works on all 459 transcripts** 

**The simplest join: each transcript and its survey row are on the same date and in the same prompt window (noon / afternoon / evening). So bucket the hour, and merge on (pid, date, slot). No nearest-time logic needed.**

In [183]:
import pandas as pd
import os, re
from docx import Document

# === 1. Reload survey data fresh ===
df_survey = pd.read_stata('data.dta')
df_survey['recordtime_dt'] = pd.to_datetime(df_survey['recordtime'], format='ISO8601')
print('df_survey loaded:', df_survey.shape)

# === 2. Load transcripts ===
TRANSCRIPT_DIR = 'Audio Transcripts'
PAT = re.compile(r'^Text (\d+)-([A-Za-z0-9]+)-(\d+)-(\d+)-(\d+)\.docx$')

def parse_name(fname):
    m = PAT.match(fname)
    if not m: return None
    ts, h, a, b, c = m.groups()
    a, b, c = int(a), int(b), int(c)
    if b == 43:   pid, survey = a, c
    elif c == 43: pid, survey = a, b
    else: return None
    return {'timestamp': ts, 'hash': h, 'participantid': pid, 'survey': survey}

rows = []
for fname in sorted(os.listdir(TRANSCRIPT_DIR)):
    if not fname.endswith('.docx'): continue
    meta = parse_name(fname)
    if meta is None: continue
    meta['filename'] = fname
    d = Document(os.path.join(TRANSCRIPT_DIR, fname))
    meta['text'] = '\n'.join(p.text for p in d.paragraphs).strip()
    meta['n_words'] = len(meta['text'].split())
    rows.append(meta)

df_tx = pd.DataFrame(rows)
df_tx['ts'] = pd.to_datetime(df_tx['timestamp'], format='%Y%m%d%H%M%S%f').dt.tz_localize('Europe/Berlin')
print('df_tx loaded:    ', df_tx.shape)

# === 3. Add date + daypart slot to BOTH ===
def slot(h):
    if h < 15: return 12      # noon
    if h < 20: return 17      # afternoon
    return 22                  # evening

df_tx['date']     = df_tx['ts'].dt.date
df_tx['slot']     = df_tx['ts'].dt.hour.map(slot)
df_survey['recordtime_dt'] = pd.to_datetime(df_survey['recordtime'], errors='coerce')
df_survey['recordtime_dt'] = pd.to_datetime(df_survey['recordtime'], errors='coerce', utc=True).dt.tz_convert('Europe/Berlin')
df_survey['date'] = df_survey['recordtime_dt'].dt.date
df_survey['slot'] = df_survey['recordtime_dt'].dt.hour.map(slot)

# === 4. Merge on (pid, date, slot) — same person, same day, same daypart ===
panel = df_survey.merge(
    df_tx[['participantid','date','slot','text','n_words','filename']],
    on=['participantid','date','slot'],
    how='left'
)

print('\n=== Phase 3 result ===')
print('Survey rows:           ', len(panel))
print('Rows with transcript:  ', panel['text'].notna().sum())
print('Transcripts available: ', len(df_tx))
print('Orphan transcripts:    ', len(set(df_tx['filename']) - set(panel['filename'].dropna())))

df_survey loaded: (956, 1179)
df_tx loaded:     (459, 8)

=== Phase 3 result ===
Survey rows:            956
Rows with transcript:   458
Transcripts available:  459
Orphan transcripts:     1


In [184]:
orphan_files = set(df_tx['filename']) - set(panel['filename'].dropna())
print(df_tx[df_tx['filename'].isin(orphan_files)][['participantid','date','slot','ts','n_words','filename']])

     participantid        date  slot                               ts  \
365         119122  2025-05-21    22 2025-05-21 22:24:21.873000+02:00   

     n_words                                           filename  
365        8  Text 20250521222421873-11f1011de4a2e7a2-119122...  


In [185]:
df_survey[(df_survey['participantid']==119122) & 
          (df_survey['recordtime_dt'].dt.date == pd.Timestamp('2025-05-21').date())]

,participantid,own_netid_cohort1,own_netid_cohort3,own_netid_cohort5,own_netid_cohort7,own_netid_master,cohort_bachelor,cohort_master,recordtime,relationship_satisfaction,...,friends_cohortmaster_45,friends_cohortmaster_46,friends_cohortmaster_47,friends_cohortmaster_48,friends_cohortmaster_49,friends_cohortmaster_50,friends_cohortmaster_51,recordtime_dt,date,slot
923,119122,,,,,22,,6 Semester,2025-05-21 12:26:52.206000+02:00,https://file.avicennaresearch.com/media/resp_f...,...,,,,,,NaN,NaN,2025-05-21 12:26:52.206000+02:00,2025-05-21,12
924,119122,,,,,22,,6 Semester,2025-05-21 17:19:14.009000+02:00,https://file.avicennaresearch.com/media/resp_f...,...,,,,,,NaN,NaN,2025-05-21 17:19:14.009000+02:00,2025-05-21,17


In [186]:
# Does pid 119122 have any survey row on 2025-05-21 evening (slot 22)?
sv = df_survey[(df_survey['participantid'] == 119122) &
               (df_survey['date'] == pd.Timestamp('2025-05-21').date()) &
               (df_survey['slot'] == 22)]
print('Survey rows for pid 119122 on 2025-05-21 evening:', len(sv))
print(sv[['participantid','recordtime','slot']])

# For context — what other pings does this person have around that date?
context = df_survey[(df_survey['participantid'] == 119122) &
                    (df_survey['date'].between(pd.Timestamp('2025-05-20').date(),
                                               pd.Timestamp('2025-05-22').date()))]
print('\nAll their pings 2025-05-20 to 2025-05-22:')
print(context[['recordtime','slot']])

Survey rows for pid 119122 on 2025-05-21 evening: 0
Empty DataFrame
Columns: [participantid, recordtime, slot]
Index: []

All their pings 2025-05-20 to 2025-05-22:
                           recordtime  slot
920  2025-05-20 12:48:13.691000+02:00    12
921  2025-05-20 17:49:05.909000+02:00    17
922  2025-05-20 22:22:55.877000+02:00    22
923  2025-05-21 12:26:52.206000+02:00    12
924  2025-05-21 17:19:14.009000+02:00    17
926  2025-05-22 12:32:21.869000+02:00    12
927  2025-05-22 18:06:19.896000+02:00    17
928  2025-05-22 22:27:38.345000+02:00    22


In [187]:
print('Total transcripts by pid 119122:', (df_tx['participantid']==119122).sum())
print('Of which matched to a survey row:', 
      panel[(panel['participantid']==119122) & panel['text'].notna()].shape[0])
print('Of which orphaned:', (df_tx['participantid']==119122).sum() 
      - panel[(panel['participantid']==119122) & panel['text'].notna()].shape[0])

Total transcripts by pid 119122: 19
Of which matched to a survey row: 18
Of which orphaned: 1


In [188]:
# 1. The orphan transcript itself
orph = df_tx[~df_tx['filename'].isin(panel['filename'].dropna())]
print('Orphan transcript:')
print(orph[['participantid','date','slot','ts','n_words','filename']].to_string(index=False))

# 2. Does df_survey have ANY row for pid 119122 on 2025-05-21 evening?
sv_evening = df_survey[(df_survey['participantid']==119122) 
                     & (df_survey['date']==pd.Timestamp('2025-05-21').date())
                     & (df_survey['slot']==22)]
print(f'\nSurvey rows for pid 119122 on 2025-05-21 evening (slot 22): {len(sv_evening)}')
print(sv_evening[['recordtime','slot']].to_string(index=False))

# 3. What survey rows does pid 119122 have on the surrounding days?
sv_window = df_survey[(df_survey['participantid']==119122) 
                    & (df_survey['date'].between(pd.Timestamp('2025-05-20').date(),
                                                 pd.Timestamp('2025-05-22').date()))]
print(f'\nAll survey rows for pid 119122 on 2025-05-20..22:')
print(sv_window[['recordtime','slot']].to_string(index=False))

# 4. Read the orphan transcript text
import os
fp = os.path.join('transcripts/Audio Transcripts', orph.iloc[0]['filename'])
print(f'\nOrphan transcript text:\n  {orph.iloc[0]["text"]}')

Orphan transcript:
 participantid       date  slot                               ts  n_words                                                     filename
        119122 2025-05-21    22 2025-05-21 22:24:21.873000+02:00        8 Text 20250521222421873-11f1011de4a2e7a2-119122-23728-43.docx

Survey rows for pid 119122 on 2025-05-21 evening (slot 22): 0
Empty DataFrame
Columns: [recordtime, slot]
Index: []

All survey rows for pid 119122 on 2025-05-20..22:
                      recordtime  slot
2025-05-20 12:48:13.691000+02:00    12
2025-05-20 17:49:05.909000+02:00    17
2025-05-20 22:22:55.877000+02:00    22
2025-05-21 12:26:52.206000+02:00    12
2025-05-21 17:19:14.009000+02:00    17
2025-05-22 12:32:21.869000+02:00    12
2025-05-22 18:06:19.896000+02:00    17
2025-05-22 22:27:38.345000+02:00    22

Orphan transcript text:
  Person 1: #00:00:00-3# Ich bin sehr zufrieden. #00:00:01-6#


Phase 3 verified end-to-end:

956 survey rows ✅

459 transcripts ✅

458 matched (one survey row + one transcript per match)

1 transcript orphaned (legitimately — no survey row to attach to)

In [189]:
panel.to_parquet('df_panel.parquet')
print('Saved:', panel.shape)

Saved: (956, 1184)


In [190]:
# === Phase 3: content check ===
import pandas as pd

# 1. Word count distribution
print('=== Word counts (all 459 transcripts) ===')
print(df_tx['n_words'].describe().round(1))
print()

# 2. Super-short transcripts (could be silence / unusable)
print('=== 10 shortest transcripts ===')
short = df_tx.nsmallest(10, 'n_words')[['participantid','date','slot','n_words','text']]
print(short.to_string(index=False))
print()

# 3. Language sniff — count typical German words vs English words
de_markers = ['ich','und','die','das','nicht','heute','war','mir','mich','sehr']
en_markers = ['the','and','was','today','very','with','have','that','this','about']

def lang_score(text):
    t = text.lower()
    de = sum(t.count(' '+w+' ') for w in de_markers)
    en = sum(t.count(' '+w+' ') for w in en_markers)
    return de, en

df_tx[['de_hits','en_hits']] = df_tx['text'].apply(lambda t: pd.Series(lang_score(t)))
print('=== Language sniff ===')
print('Transcripts where English markers > German markers:', 
      (df_tx['en_hits'] > df_tx['de_hits']).sum())
print('Distribution of de_hits:')
print(df_tx['de_hits'].describe().round(1))
print('Distribution of en_hits:')
print(df_tx['en_hits'].describe().round(1))

# Show any English-leaning transcripts
suspicious = df_tx[df_tx['en_hits'] > df_tx['de_hits']]
if len(suspicious) > 0:
    print('\n=== English-leaning samples ===')
    print(suspicious[['participantid','n_words','text']].head(10).to_string(index=False))

=== Word counts (all 459 transcripts) ===
count    459.0
mean      26.8
std       36.9
min        1.0
25%        6.0
50%       11.0
75%       35.0
max      293.0
Name: n_words, dtype: float64

=== 10 shortest transcripts ===
 participantid       date  slot  n_words         text
        110917 2025-01-27    17        1 #00:00:00-0#
        110975 2025-01-27    22        1 #00:00:00-0#
        110917 2025-01-27    22        1 #00:00:00-0#
        110917 2025-01-28    12        1 #00:00:00-0#
        110931 2025-01-28    12        1 #00:00:00-0#
        110917 2025-01-28    17        1 #00:00:00-0#
        110975 2025-01-28    22        1 #00:00:00-0#
        110917 2025-01-28    22        1 #00:00:00-0#
        110960 2025-01-29    12        1 #00:00:00-0#
        110917 2025-01-29    12        1 #00:00:00-0#

=== Language sniff ===
Transcripts where English markers > German markers: 0
Distribution of de_hits:
count    459.0
mean       4.5
std        6.9
min        0.0
25%        0.0
50%

Word counts
Median diary: 11 words (about one sentence)

75th percentile: 35 words

Longest: 293 words

Shortest: 1 "word" — which is actually just #00:00:00-0# (a timestamp marker with no real speech)

Language
All German — no English-leaning entries. ✅

The 1-word transcripts are silent recordings
Those #00:00:00-0# entries mean the participant opened the audio recorder but said nothing (or the audio was empty). The transcription service produced a docx but with only the timestamp marker — no spoken words.

##### **Next step: clean and re-count**
Strip the Person 1: tag and the #00:00:XX-X# timestamp markers, then recount words on actual spoken content.

In [191]:
import re

def clean_text(t):
    # Remove Person N: tags
    t = re.sub(r'Person\s*\d+\s*:', ' ', t)
    # Remove #HH:MM:SS-S# timestamps
    t = re.sub(r'#\d{2}:\d{2}:\d{2}-\d+#', ' ', t)
    # Collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_tx['text_clean']     = df_tx['text'].apply(clean_text)
df_tx['n_words_clean']  = df_tx['text_clean'].str.split().str.len()

print('=== Word counts AFTER cleaning ===')
print(df_tx['n_words_clean'].describe().round(1))

print('\n=== Silent recordings (0 spoken words after cleaning) ===')
silent = df_tx[df_tx['n_words_clean'] == 0]
print(f'Count: {len(silent)}')
print(silent[['participantid','date','slot','text']].head(15).to_string(index=False))

print('\n=== Recordings with very few spoken words (1–3) ===')
print(df_tx[df_tx['n_words_clean'].between(1,3)]
      [['participantid','date','slot','n_words_clean','text_clean']].head(10).to_string(index=False))

=== Word counts AFTER cleaning ===
count    459.0
mean      23.0
std       36.7
min        0.0
25%        2.0
50%        7.0
75%       31.0
max      289.0
Name: n_words_clean, dtype: float64

=== Silent recordings (0 spoken words after cleaning) ===
Count: 27
 participantid       date  slot         text
        110917 2025-01-27    17 #00:00:00-0#
        110975 2025-01-27    22 #00:00:00-0#
        110917 2025-01-27    22 #00:00:00-0#
        110917 2025-01-28    12 #00:00:00-0#
        110931 2025-01-28    12 #00:00:00-0#
        110917 2025-01-28    17 #00:00:00-0#
        110975 2025-01-28    22 #00:00:00-0#
        110917 2025-01-28    22 #00:00:00-0#
        110960 2025-01-29    12 #00:00:00-0#
        110917 2025-01-29    12 #00:00:00-0#
        110946 2025-01-29    17 #00:00:00-0#
        110975 2025-01-29    22 #00:00:00-0#
        110917 2025-01-30    12 #00:00:00-0#
        110960 2025-01-30    12 #00:00:00-0#
        110960 2025-01-30    17 #00:00:00-0#

=== Recordings with

The cleaning works:

27 silent recordings (opened recorder, said nothing) — 6% of all transcripts

The 1–3 word transcripts are short but meaningful: "Zufrieden.", "Sehr zufrieden." — direct one-word answers to the prompt

Decision: keep or drop the 27 silent ones?
Standard practice: keep them, flag them, document. Then exclude from any analysis that needs text content. That way:

        We don't pretend they don't exist

        The full panel stays intact

        Reports/models that need text can filter them out cleanly

But first — let's check if any one participant produced most of the silent recordings. If one person was always silent, they may need special handling.

In [192]:
# Silent rate per participant
silent_per_pid = df_tx.assign(is_silent=df_tx['n_words_clean']==0)\
                      .groupby('participantid').agg(
                          total=('filename','count'),
                          silent=('is_silent','sum'))
silent_per_pid['silent_rate'] = (silent_per_pid['silent'] / silent_per_pid['total']).round(2)
print('Participants with silent recordings (sorted by rate):')
print(silent_per_pid[silent_per_pid['silent']>0].sort_values('silent_rate', ascending=False))

Participants with silent recordings (sorted by rate):
               total  silent  silent_rate
participantid                            
110917            12      11         0.92
110960            14       6         0.43
110975            19       4         0.21
110931             7       1         0.14
110946            15       2         0.13
110918            18       2         0.11
119116            18       1         0.06


Important finding. One participant stands out:

pid	total	silent	silent rate
    110917	12	11	92% ⚠️
    110960	14	6	43%
    Others	—	—	≤ 21% (normal variation)

110917 is effectively a non-participant for text analysis. 

Out of 12 audio diaries, only 1 contains any speech. Let's look at it before deciding:

In [193]:
# What did pid 110917 actually say in their one non-silent recording?
non_silent_110917 = df_tx[(df_tx['participantid']==110917) & (df_tx['n_words_clean']>0)]
print(non_silent_110917[['date','slot','n_words_clean','text_clean']].to_string(index=False))

      date  slot  n_words_clean    text_clean
2025-01-30    22              3 Ist das denn?


On the 110917 transcript: confirmed
"Ist das denn?" translates to roughly "Is that it then?" — that's not a diary entry. It's someone muttering while testing the recorder. So pid 110917 effectively has zero usable diary content. Confirmed exclusion for text analyses.

Suggested handling
For text-based analyses (LLM measure, NLP benchmarks, semantic features): exclude pid 110917 entirely — they don't have enough text to compute anything meaningful.

For survey-only analyses (UCLA × PHQ etc.): keep their baseline scores.

For panel models linking text to scores: their text contribution is essentially zero, so they fall out naturally.

This needs a single sentence in the Methods section:

"One participant (PID 110917) was excluded from text-based analyses due to a 92% rate of silent recordings (1 of 12 diaries contained speech). They were retained in baseline survey analyses."



In [194]:
# Flag silent recordings and the no-text participant
df_tx['is_silent']  = (df_tx['n_words_clean'] == 0)
df_tx['exclude_text'] = df_tx['participantid'].isin([110917])

print('Silent recordings:        ', df_tx['is_silent'].sum())
print('From pid 110917:          ', df_tx['exclude_text'].sum())
print('Usable text transcripts:  ', ((~df_tx['is_silent']) & (~df_tx['exclude_text'])).sum())

Silent recordings:         27
From pid 110917:           12
Usable text transcripts:   431


Exact numbers now:

                            Count
    Total transcripts	459
    Silent	                27

From pid 110917 (excluded)	12 (11 silent + 1 test utterance)
    
    
Usable for text analyses	431

In [195]:
# Re-merge with the cleaned & flagged text columns so panel has everything
panel = df_survey.merge(
    df_tx[['participantid','date','slot','text_clean','n_words_clean',
           'is_silent','exclude_text','filename']],
    on=['participantid','date','slot'], how='left'
)
panel.to_parquet('df_panel.parquet')

print('Panel saved:', panel.shape)
print('Rows with linked transcript:                 ', panel['filename'].notna().sum())
print('Rows with USABLE text (non-silent, included):', 
      ((panel['filename'].notna()) & (~panel['is_silent'].fillna(False)) 
       & (~panel['exclude_text'].fillna(False))).sum())

Panel saved: (956, 1186)
Rows with linked transcript:                  458
Rows with USABLE text (non-silent, included): 430


| Level                              | Count | What it is                                           |
| ---------------------------------- | ----- | ---------------------------------------------------- |
| Transcripts (text-level)           | 459   | All loaded                                           |
| Non-silent transcripts             | 432   | 459 − 27 silent                                      |
| Non-silent & not from excluded pid | 431   | 432 − 1 recorder-test from pid 110917                |
| Usable panel rows                  | 430   | 431 − 1 orphan (no matching survey row to attach to) |


So 431 = transcripts we'd use for text-only analyses (where we don't need scores). 

430 = panel rows we'd use for any analysis that needs both text and scores (the main study design).

In [196]:
# 1. Counts: how many matched, how many orphans
print('Transcripts total:', len(df_tx))
print('Matched to a survey row:', panel['filename'].notna().sum())
print('Orphans:', len(set(df_tx['filename']) - set(panel['filename'].dropna())))

# 2. Did any transcript land on MORE than one survey row? (Would mean duplicate matches)
dup_per_tx = panel.dropna(subset=['filename']).groupby('filename').size()
print('\nTranscripts attached to >1 survey row:', (dup_per_tx > 1).sum())

# 3. Did any survey row receive MORE than one transcript? (Many-to-one)
dup_per_row = panel.dropna(subset=['filename']).groupby(['participantid','date','slot']).size()
print('Survey rows holding >1 transcript:', (dup_per_row > 1).sum())

# 4. Spot-check: pick 5 random matches and show timestamps side-by-side
print('\nSpot-check (transcript timestamp vs survey recordtime):')
sample = panel.dropna(subset=['filename']).sample(5, random_state=0)
# pull each transcript's ts from df_tx
sample = sample.merge(df_tx[['filename','ts']], on='filename')
print(sample[['participantid','date','slot','ts','recordtime']].to_string(index=False))

Transcripts total: 459
Matched to a survey row: 458
Orphans: 1

Transcripts attached to >1 survey row: 0
Survey rows holding >1 transcript: 0

Spot-check (transcript timestamp vs survey recordtime):
 participantid       date  slot                               ts                       recordtime
        110981 2025-01-30    12 2025-01-30 12:20:07.363000+01:00 2025-01-30 12:20:15.482000+01:00
        111278 2025-01-29    12 2025-01-29 12:43:14.862000+01:00 2025-01-29 12:43:53.429000+01:00
        119079 2025-05-24    22 2025-05-24 22:19:03.131000+02:00 2025-05-24 22:19:08.506000+02:00
        111267 2025-01-31    17 2025-01-31 17:11:53.663000+01:00 2025-01-31 17:12:15.294000+01:00
        110975 2025-01-27    22 2025-01-27 22:15:55.206000+01:00 2025-01-27 22:16:07.105000+01:00


What we're looking at:

How many usable rows each person contributes

Any participant too sparse (e.g. only 1–2 rows) for panel models

Distribution shape (is the data balanced across people, or dominated by a few high-contributors?)

In [197]:
# === Phase 4.1: analysable subset ===
mask = (panel['filename'].notna()             # has a transcript
        & ~panel['is_silent'].fillna(False)   # not silent
        & ~panel['exclude_text'].fillna(False))  # participant not excluded

panel_text = panel[mask].copy()
panel_text.to_parquet('panel_textonly.parquet')

print('Usable panel shape:', panel_text.shape)
print('Unique participants:', panel_text['participantid'].nunique())
print()

# Per-participant counts
per_pid = panel_text.groupby('participantid').agg(
    n_rows=('filename','count'),
    n_words_mean=('n_words_clean','mean'),
    n_words_total=('n_words_clean','sum'),
).round(1).sort_values('n_rows')

print('Per-participant counts (sorted by # of usable rows):')
print(per_pid.to_string())
print()
print('Distribution of rows per participant:')
print(per_pid['n_rows'].describe().round(1))

Usable panel shape: (430, 1186)
Unique participants: 40

Per-participant counts (sorted by # of usable rows):
               n_rows  n_words_mean  n_words_total
participantid                                     
110912              1          39.0           39.0
119108              1          41.0           41.0
110971              1           4.0            4.0
110950              1           4.0            4.0
110907              2          16.0           32.0
110982              2          12.0           24.0
110959              2          48.5           97.0
110961              2           2.0            4.0
110941              2          38.5           77.0
119087              2           9.5           19.0
110958              3          82.3          247.0
111258              3          74.0          222.0
110916              4          25.5          102.0
110924              4           2.0            8.0
110954              4           8.8           35.0
110931              6  

 **Row count per participant**

Median: 13 rows (good)

4 participants with only 1 row (110912, 119108, 110971, 110950)

6 with 2 rows, 2 with 3 rows

12 of 40 (~30%) have ≤3 rows

**Word count per recording (more interesting)**

Some participants record almost every prompt but consistently say just one word:

| pid    | rows | avg words / recording |
| ------ | ---- | --------------------- |
| 110973 | 18   | 1.0                   |
| 119116 | 17   | 1.2                   |
| 110946 | 13   | 1.5                   |
| 110975 | 15   | 1.7                   |
| 110961 | 2    | 2.0                   |
| 119091 | 20   | 2.0                   |


These are people who almost always answer with just "Zufrieden." — technically not silent, but offering minimal linguistic content. NLP features will be unstable on them.

**On the other end:**

pid 119109: 142 words/recording (very expressive, 16 rows)

pid 111295, 111287: ~70 words/recording

Methodological options
Three honest choices for how to handle this:
| Option                                                    | What it does                                 | Trade-off                                                                             |
| --------------------------------------------------------- | -------------------------------------------- | ------------------------------------------------------------------------------------- |
| A. Keep everyone (panel models with random intercepts)    | All 40 participants × 430 rows               | Models handle uneven N per person via shrinkage; minimal-word people contribute noise |
| B. Minimum-rows threshold (e.g. ≥ 3 rows)                 | Drop ~10 participants                        | Cleaner panel but loses people                                                        |
| C. Minimum-words threshold per recording (e.g. ≥ 5 words) | Drop short recordings, keep all participants | Each row needs real text to be analysed; some participants effectively drop out       |

given Supervisor's "panel data with all 459 obs" guidance: 

**Option A** — keep everyone, but add a continuous n_words_clean covariate to the models so we can statistically control for text length. We can also do sensitivity analyses restricting to rows with ≥5 words and check if results hold.

In [199]:
import numpy as np
import pandas as pd

# ============================================
# Phase 2: Re-apply scoring to fresh df_survey
# ============================================

# --- Response mappings ---
UCLA_MAP = {'Stimmt gar nicht':1,'Stimmt wenig':2,'Stimmt teils-teils':3,'Stimmt ziemlich':4,'Stimmt völlig':5,'':np.nan}
BSSS_MAP = {'Stimme überhaupt nicht zu':1,'Stimme eher nicht zu':2,'Stimme etwas zu':3,'Stimme voll und ganz zu':4,'':np.nan}
PHQ9_MAP = {'Überhaupt nicht':0,'An einzelnen Tagen':1,'An mehr als der Hälfte der Tage':2,'Beinahe jeden Tag':3,'':np.nan}
GAD7_MAP = PHQ9_MAP
LON_MAP  = {'0 gar nicht':0,'1':1,'2':2,'3':3,'4 sehr':4,'':np.nan}
STRESS_MAP = LON_MAP

def map_cols(df, cols, mapping):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(mapping)

# --- UCLA (trait, 1-5) ---
ucla_cols = [f'einsam_{i}' for i in range(1,21)]
map_cols(df_survey, ucla_cols, UCLA_MAP)
REVERSE_UCLA = {1,2,4,5,8,9,14,15,18,19}  # data-driven, validated earlier
ucla_scored = df_survey[ucla_cols].copy()
for i in REVERSE_UCLA:
    ucla_scored[f'einsam_{i}'] = 6 - ucla_scored[f'einsam_{i}']
df_survey['UCLA_total'] = ucla_scored.sum(axis=1, min_count=20)

# --- BSSS (1-4) ---
bsss_cols = [f'social_support_{i}' for i in range(1,9)]
map_cols(df_survey, bsss_cols, BSSS_MAP)
df_survey['BSSS_total'] = df_survey[bsss_cols].sum(axis=1, min_count=8)
df_survey['BSSS_mean']  = df_survey[bsss_cols].mean(axis=1)

# --- PHQ-9 (0-3) with prorating for 1 missing item ---
phq_cols = [f'depression_{i}' for i in range(1,10)]
map_cols(df_survey, phq_cols, PHQ9_MAP)
phq = df_survey[phq_cols]
n_present = phq.notna().sum(axis=1)
phq_mean_per_row = phq.mean(axis=1)
df_survey['PHQ9_total'] = np.where(n_present >= 8, phq_mean_per_row * 9, np.nan)

# --- GAD-7 (0-3) ---
gad_cols = [f'anxiety_{i}' for i in range(1,8)]
map_cols(df_survey, gad_cols, GAD7_MAP)
df_survey['GAD7_total'] = df_survey[gad_cols].sum(axis=1, min_count=7)

# --- Mood PA / NA (0-100 slider; numeric already) ---
mood_cols = [f'mood_{i}' for i in range(1,12)]
for c in mood_cols:
    df_survey[c] = pd.to_numeric(df_survey[c], errors='coerce')
df_survey['mood_PA'] = df_survey[['mood_2','mood_4','mood_5','mood_6']].mean(axis=1)
df_survey['mood_NA'] = df_survey[['mood_1','mood_3','mood_7','mood_8','mood_9']].mean(axis=1)

# --- State loneliness (0-4, reverse {5,6,7,8}) ---
lon_cols = [f'loneliness_{i}' for i in range(1,9)]
map_cols(df_survey, lon_cols, LON_MAP)
lon_scored = df_survey[lon_cols].copy()
for i in [5,6,7,8]:
    lon_scored[f'loneliness_{i}'] = 4 - lon_scored[f'loneliness_{i}']
df_survey['loneliness_state'] = lon_scored.mean(axis=1)

# --- Study stress (2 items, reverse #1) ---
df_survey['stress_uni_1'] = df_survey['stress_uni_1'].map(STRESS_MAP)
df_survey['stress_uni_2'] = df_survey['stress_uni_2'].map(STRESS_MAP)
df_survey['stress_uni'] = ((4 - df_survey['stress_uni_1']) + df_survey['stress_uni_2']) / 2

# --- Forward-fill baseline (trait) scores within participant ---
for col in ['UCLA_total','BSSS_total','BSSS_mean','PHQ9_total','GAD7_total']:
    df_survey[col] = df_survey.groupby('participantid')[col].transform('first')

print('Phase 2 re-scored. df_survey shape:', df_survey.shape)
print(df_survey[['UCLA_total','BSSS_total','PHQ9_total','GAD7_total',
                 'mood_PA','mood_NA','loneliness_state','stress_uni']].describe().round(2))

Phase 2 re-scored. df_survey shape: (956, 1190)
       UCLA_total  BSSS_total  PHQ9_total  GAD7_total  mood_PA  mood_NA  \
count      956.00      956.00      956.00      956.00   956.00   956.00   
mean        40.06       28.74        9.49        7.81    46.04    21.51   
std         11.73        3.89        4.48        4.31    20.32    18.65   
min         22.00       18.00        1.00        1.00     0.00     0.00   
25%         31.00       27.00        6.75        5.00    30.75     6.60   
50%         37.00       30.00        8.00        7.00    44.38    16.50   
75%         46.00       32.00       13.00       11.00    61.25    31.80   
max         71.00       32.00       20.00       18.00    97.50    95.00   

       loneliness_state  stress_uni  
count            954.00      948.00  
mean               0.65        2.24  
std                0.67        0.98  
min                0.00        0.00  
25%                0.12        1.50  
50%                0.50        2.00  
75%       

In [200]:
panel = df_survey.merge(
    df_tx[['participantid','date','slot','text_clean','n_words_clean',
           'is_silent','exclude_text','filename']],
    on=['participantid','date','slot'], how='left'
)

mask = (panel['filename'].notna()
        & ~panel['is_silent'].fillna(False)
        & ~panel['exclude_text'].fillna(False))
panel_text = panel[mask].copy()
panel_text.to_parquet('panel_textonly.parquet')

print('panel_text shape:', panel_text.shape)

panel_text shape: (430, 1195)


In [201]:
baseline = df_survey.groupby('participantid')[['UCLA_total','BSSS_total','PHQ9_total','GAD7_total']].first()
print(baseline.describe().round(2))

       UCLA_total  BSSS_total  PHQ9_total  GAD7_total
count       55.00       55.00       55.00       55.00
mean        39.98       28.67        9.43        7.76
std         11.55        3.85        4.52        4.29
min         22.00       18.00        1.00        1.00
25%         31.00       27.00        6.38        4.50
50%         37.00       30.00        8.00        7.00
75%         46.00       32.00       12.50       11.00
max         71.00       32.00       20.00       18.00


In [203]:
# Save the scored survey AND the merged panel — durable, no more memory-only state
df_survey.to_parquet('df_survey_scored.parquet')
panel.to_parquet('df_panel.parquet')
panel_text.to_parquet('panel_textonly.parquet')
print('Saved:')
print('  df_survey_scored.parquet:', df_survey.shape)
print('  df_panel.parquet:        ', panel.shape)
print('  panel_textonly.parquet:  ', panel_text.shape)

Saved:
  df_survey_scored.parquet: (956, 1190)
  df_panel.parquet:         (956, 1195)
  panel_textonly.parquet:   (430, 1195)


In [205]:
print('=== Usable panel score distributions ===')
print(panel_text[['mood_PA','mood_NA','loneliness_state','stress_uni',
                  'UCLA_total','BSSS_total','PHQ9_total','GAD7_total']].describe().round(2))

print('\n=== Word count distribution on usable subset ===')
print(panel_text['n_words_clean'].describe().round(1))
print('\nQuantiles for the >=5-word sensitivity threshold:')
print('Rows with n_words >= 5:', (panel_text['n_words_clean']>=5).sum())
print('Rows with n_words < 5: ', (panel_text['n_words_clean']<5).sum())

=== Usable panel score distributions ===
       mood_PA  mood_NA  loneliness_state  stress_uni  UCLA_total  BSSS_total  \
count   430.00   430.00            430.00      430.00      430.00      430.00   
mean     49.12    23.47              0.62        2.38       39.26       28.53   
std      21.70    20.90              0.70        0.95       12.46        4.29   
min       0.00     0.00              0.00        0.00       22.00       18.00   
25%      33.25     5.00              0.12        1.50       30.00       27.00   
50%      47.38    19.00              0.38        2.50       38.00       31.00   
75%      67.19    36.35              1.00        3.00       42.00       32.00   
max      97.50    95.00              3.12        4.00       71.00       32.00   

       PHQ9_total  GAD7_total  
count      430.00      430.00  
mean        10.20        8.33  
std          4.98        4.70  
min          1.00        1.00  
25%          7.00        4.00  
50%          8.00        8.00  
75%  

What this tells us
Trait scores (UCLA, BSSS, PHQ-9, GAD-7) — slightly different from baseline (39.98 → 39.26 etc.) because some of the 55 people didn't contribute usable transcripts, and people with more pings get counted more times. That's expected and correct for the panel view.

State scores (per-ping)

mood_PA mean ≈ 49 (range 0–97) → wide spread, good signal

mood_NA mean ≈ 23 (range 0–95) → skewed low, normal for non-clinical sample

loneliness_state mean ≈ 0.62 → most pings are low loneliness (floor effect, expected)

stress_uni mean ≈ 2.38 → mid-range, healthy variance

Word counts — this is the important one

Median = 10.5 words

Mean = 24.5 (pulled up by the 289-word outlier)

252 rows have ≥5 words, 178 rows have <5 words

That ~40% of rows being very short is the real story. Many diary entries are basically one-liners ("Mir geht's gut" type). That's why Heike's n_words_clean as covariate + sensitivity at ≥5 words matters — you'll need to show that results aren't driven by ultra-short entries.

In [209]:
from openai import AzureOpenAI
import requests

# === Keys (from Heike's and Prof. Grund's emails) ===
AZURE_KEY = "c9f4bcf2ea354bbc8e9dbd9ceccffbb8"
AZURE_ENDPOINT = "https://soziologieaitutor.openai.azure.com/"
AZURE_DEPLOYMENT = "gpt-5-chat"

MERTON_URL = "http://134.130.156.55:8080"
MERTON_KEY = " 9f2b88dc-fb9e-4b2d-9a91-3d65f07a1ef4"

# === Test 1: Azure GPT-5-chat ===
azure_client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_KEY,
)
r = azure_client.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Reply with exactly: OK"}
    ],
    max_tokens=10,
    model=AZURE_DEPLOYMENT,
)
print("AZURE  →", r.choices[0].message.content)

# === Test 2: Merton Gemma-3:1b ===
headers = {"Authorization": f"Bearer {MERTON_KEY}", "Content-Type": "application/json"}
payload = {"query": "Reply with exactly: OK"}
resp = requests.post(f"{MERTON_URL}/query", json=payload, headers=headers, timeout=60)
print("MERTON →", resp.status_code, resp.json() if resp.status_code == 200 else resp.text)

AZURE  → OK


ConnectionError: HTTPConnectionPool(host='134.130.156.55', port=8080): Max retries exceeded with url: /query (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001BAF3582C80>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))